## Feature Engineering Pipeline

Merges cleaned AURN pollution data with Open-Meteo weather data, adds
time features, lag features, rolling means, and a 24-hour-ahead PM2.5
target column. Produces `data/processed/features_engineered.csv`.

In [1]:
import pathlib
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)

In [2]:
# Cell 1 -- load cleaned AURN and weather datasets
PROC = pathlib.Path('../data/processed')

aurn    = pd.read_csv(PROC / 'aurn_cleaned.csv',      parse_dates=['datetime'], index_col='datetime')
weather = pd.read_csv(PROC / 'weather_historical.csv', parse_dates=['datetime'], index_col='datetime')

print('AURN shape    :', aurn.shape,    ' | cities:', sorted(aurn['city'].unique()))
print('Weather shape :', weather.shape, ' | cities:', sorted(weather['city'].unique()))
aurn.head(3)

AURN shape    : (87725, 4)  | cities: ['Birmingham A4540 Roadside', 'Edinburgh St Leonards', 'Leeds Centre', 'London Marylebone Road', 'Manchester Piccadilly']
Weather shape : (87720, 7)  | cities: ['Birmingham', 'Edinburgh', 'Leeds', 'London', 'Manchester']


,city,o3,no2,pm25
datetime,,,,
2023-01-01 01:00:00,Birmingham A4540 Roadside,61.33451,14.17608,7.123
2023-01-01 02:00:00,Birmingham A4540 Roadside,65.79158,10.04362,4.057
2023-01-01 03:00:00,Birmingham A4540 Roadside,66.40692,11.48356,4.363


In [3]:
# Cell 2 -- merge AURN + weather on (city, datetime)
# Rename weather city labels to match AURN station names
CITY_MAP = {
    'London':     'London Marylebone Road',
    'Birmingham': 'Birmingham A4540 Roadside',
    'Manchester': 'Manchester Piccadilly',
    'Leeds':      'Leeds Centre',
    'Edinburgh':  'Edinburgh St Leonards',
}
weather = weather.reset_index()
weather['city'] = weather['city'].map(CITY_MAP)
weather = weather.set_index('datetime')

aurn_r    = aurn.reset_index()
weather_r = weather.reset_index()

df = pd.merge(
    aurn_r,
    weather_r,
    on=['datetime', 'city'],
    how='inner',
)
df = df.sort_values(['city', 'datetime']).reset_index(drop=True)

print(f'Merged shape : {df.shape}')
print(f'Date range   : {df["datetime"].min()}  to  {df["datetime"].max()}')
print(f'Cities       : {sorted(df["city"].unique())}')
df.head(3)

Merged shape : (87715, 11)
Date range   : 2023-01-01 01:00:00  to  2024-12-31 23:00:00
Cities       : ['Birmingham A4540 Roadside', 'Edinburgh St Leonards', 'Leeds Centre', 'London Marylebone Road', 'Manchester Piccadilly']


,datetime,city,o3,no2,pm25,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation,surface_pressure
0,2023-01-01 01:00:00,Birmingham A4540 Roadside,61.33451,14.17608,7.123,8.5,89,29.2,218,0.0,981.7
1,2023-01-01 02:00:00,Birmingham A4540 Roadside,65.79158,10.04362,4.057,8.0,87,27.3,222,0.0,982.7
2,2023-01-01 03:00:00,Birmingham A4540 Roadside,66.40692,11.48356,4.363,7.3,88,25.5,220,0.0,983.5


In [4]:
# Cell 3 -- time features
df['hour']       = df['datetime'].dt.hour
df['day_of_week']= df['datetime'].dt.dayofweek   # 0=Monday
df['month']      = df['datetime'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print('Time features added: hour, day_of_week, month, is_weekend')
df[['datetime','city','hour','day_of_week','month','is_weekend']].head(6)

Time features added: hour, day_of_week, month, is_weekend


,datetime,city,hour,day_of_week,month,is_weekend
0,2023-01-01 01:00:00,Birmingham A4540 Roadside,1,6,1,1
1,2023-01-01 02:00:00,Birmingham A4540 Roadside,2,6,1,1
2,2023-01-01 03:00:00,Birmingham A4540 Roadside,3,6,1,1
3,2023-01-01 04:00:00,Birmingham A4540 Roadside,4,6,1,1
4,2023-01-01 05:00:00,Birmingham A4540 Roadside,5,6,1,1
5,2023-01-01 06:00:00,Birmingham A4540 Roadside,6,6,1,1


In [5]:
# Cell 4 -- lag features and rolling means (per city)
# Sort within each city before computing lags
df = df.sort_values(['city', 'datetime']).reset_index(drop=True)

LAG_HOURS    = [1, 2, 3, 24]
ROLLING_WINS = [24, 72]

for lag in LAG_HOURS:
    df[f'pm25_lag_{lag}'] = (
        df.groupby('city')['pm25'].shift(lag)
    )

for win in ROLLING_WINS:
    df[f'pm25_roll_{win}h'] = (
        df.groupby('city')['pm25']
        .transform(lambda s: s.shift(1).rolling(win, min_periods=win // 2).mean())
    )

lag_cols = [f'pm25_lag_{l}' for l in LAG_HOURS] + [f'pm25_roll_{w}h' for w in ROLLING_WINS]
print('Lag / rolling features added:')
print(lag_cols)
df[['datetime','city','pm25'] + lag_cols].head(6)

Lag / rolling features added:
['pm25_lag_1', 'pm25_lag_2', 'pm25_lag_3', 'pm25_lag_24', 'pm25_roll_24h', 'pm25_roll_72h']


,datetime,city,pm25,pm25_lag_1,pm25_lag_2,pm25_lag_3,pm25_lag_24,pm25_roll_24h,pm25_roll_72h
0,2023-01-01 01:00:00,Birmingham A4540 Roadside,7.123,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-01 02:00:00,Birmingham A4540 Roadside,4.057,7.123,NaN,NaN,NaN,NaN,NaN
2,2023-01-01 03:00:00,Birmingham A4540 Roadside,4.363,4.057,7.123,NaN,NaN,NaN,NaN
3,2023-01-01 04:00:00,Birmingham A4540 Roadside,4.127,4.363,4.057,7.123,NaN,NaN,NaN
4,2023-01-01 05:00:00,Birmingham A4540 Roadside,5.024,4.127,4.363,4.057,NaN,NaN,NaN
5,2023-01-01 06:00:00,Birmingham A4540 Roadside,5.778,5.024,4.127,4.363,NaN,NaN,NaN


In [6]:
# Cell 5 -- target variable: next-24-hour mean PM2.5 per city
# Shift pm25 backward by 24 positions within each city group
df['pm25_next24h'] = df.groupby('city')['pm25'].shift(-24)

valid = df['pm25_next24h'].notna().sum()
total = len(df)
print(f'Target column pm25_next24h: {valid:,} valid rows out of {total:,} ({100*valid/total:.1f}%)')
df[['datetime','city','pm25','pm25_next24h']].tail(6)

Target column pm25_next24h: 81,028 valid rows out of 87,715 (92.4%)


,datetime,city,pm25,pm25_next24h
87709,2024-12-31 18:00:00,Manchester Piccadilly,8.632,NaN
87710,2024-12-31 19:00:00,Manchester Piccadilly,6.509,NaN
87711,2024-12-31 20:00:00,Manchester Piccadilly,6.179,NaN
87712,2024-12-31 21:00:00,Manchester Piccadilly,4.292,NaN
87713,2024-12-31 22:00:00,Manchester Piccadilly,3.561,NaN
87714,2024-12-31 23:00:00,Manchester Piccadilly,3.679,NaN


In [7]:
# Cell 6 -- drop rows missing the target or all lag features, report shape
before = len(df)
df = df.dropna(subset=['pm25_next24h'])
after  = len(df)
print(f'Dropped {before - after:,} rows with no target (last 24 h per city)')
print(f'Final dataset shape: {df.shape}')

missing_pct = df.isnull().mean().sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
if len(missing_pct):
    print('Columns still with NaN:')
    print((missing_pct * 100).round(2).to_string())
else:
    print('No NaN in any column after dropping missing targets.')
df.head(3)

Dropped 6,687 rows with no target (last 24 h per city)
Final dataset shape: (81028, 22)


Columns still with NaN:
no2              2.96
o3               2.35
pm25_lag_24      2.01
pm25_lag_3       1.49
pm25_lag_2       1.47
pm25_lag_1       1.44
pm25             1.41
pm25_roll_72h    1.28
pm25_roll_24h    1.04


,datetime,city,o3,no2,pm25,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation,surface_pressure,hour,day_of_week,month,is_weekend,pm25_lag_1,pm25_lag_2,pm25_lag_3,pm25_lag_24,pm25_roll_24h,pm25_roll_72h,pm25_next24h
0,2023-01-01 01:00:00,Birmingham A4540 Roadside,61.33451,14.17608,7.123,8.5,89,29.2,218,0.0,981.7,1,6,1,1,NaN,NaN,NaN,NaN,NaN,NaN,4.811
1,2023-01-01 02:00:00,Birmingham A4540 Roadside,65.79158,10.04362,4.057,8.0,87,27.3,222,0.0,982.7,2,6,1,1,7.123,NaN,NaN,NaN,NaN,NaN,4.623
2,2023-01-01 03:00:00,Birmingham A4540 Roadside,66.40692,11.48356,4.363,7.3,88,25.5,220,0.0,983.5,3,6,1,1,4.057,7.123,NaN,NaN,NaN,NaN,4.670


In [8]:
# Cell 7 -- final column summary
feature_cols = [
    'o3', 'no2', 'pm25',
    'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
    'wind_direction_10m', 'precipitation', 'surface_pressure',
    'hour', 'day_of_week', 'month', 'is_weekend',
    'pm25_lag_1', 'pm25_lag_2', 'pm25_lag_3', 'pm25_lag_24',
    'pm25_roll_24h', 'pm25_roll_72h',
]
target_col = 'pm25_next24h'
id_cols    = ['datetime', 'city']

print(f'Feature columns ({len(feature_cols)}):')
print(feature_cols)
print(f'Target          : {target_col}')
print(f'Total features  : {len(feature_cols)}')
print(f'Final shape     : {df.shape}')

Feature columns (19):
['o3', 'no2', 'pm25', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'wind_direction_10m', 'precipitation', 'surface_pressure', 'hour', 'day_of_week', 'month', 'is_weekend', 'pm25_lag_1', 'pm25_lag_2', 'pm25_lag_3', 'pm25_lag_24', 'pm25_roll_24h', 'pm25_roll_72h']
Target          : pm25_next24h
Total features  : 19
Final shape     : (81028, 22)


In [9]:
# Cell 8 -- save to data/processed/features_engineered.csv
OUT_PATH = pathlib.Path('../data/processed/features_engineered.csv')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUT_PATH, index=False)
print(f'Saved {len(df):,} rows x {len(df.columns)} columns to {OUT_PATH}')
df.tail(3)

Saved 81,028 rows x 22 columns to ..\data\processed\features_engineered.csv


,datetime,city,o3,no2,pm25,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation,surface_pressure,hour,day_of_week,month,is_weekend,pm25_lag_1,pm25_lag_2,pm25_lag_3,pm25_lag_24,pm25_roll_24h,pm25_roll_72h,pm25_next24h
87688,2024-12-30 21:00:00,Manchester Piccadilly,46.89895,25.34049,4.080,8.2,87,9.7,183,0.0,1014.5,21,0,12,0,3.915,4.670,8.538,5.825,3.717583,7.547833,4.292
87689,2024-12-30 22:00:00,Manchester Piccadilly,48.94454,23.28178,7.146,8.6,86,12.0,188,0.0,1014.4,22,0,12,0,4.080,3.915,4.670,5.094,3.644875,7.369306,3.561
87690,2024-12-30 23:00:00,Manchester Piccadilly,45.20261,24.86430,3.420,8.9,83,14.0,193,0.0,1013.7,23,0,12,0,7.146,4.080,3.915,3.986,3.730375,7.244819,3.679
